# Análise Exploratória do WikiArt

Este notebook documenta a etapa de **análise exploratória e curadoria inicial** do dataset `huggan/wikiart` utilizada no TCC.

## Objetivos

1. Inspecionar a estrutura e os metadados do WikiArt.
2. Quantificar a distribuição dos estilos artísticos.
3. Investigar a distribuição de artistas e as limitações relacionadas a `Unknown Artist`.
4. Analisar a distribuição dos gêneros temáticos.
5. Consolidar as quatro macroclasses definidas no escopo do projeto:
   - Renaissance
   - Baroque
   - Impressionism
   - Cubism
6. Identificar associações entre gênero temático e macroclasse que possam favorecer *shortcut learning*.

> Este notebook é destinado à **EDA**. A seleção definitiva das pinturas, auditoria de integridade, balanceamento e divisão em treino/validação/teste serão realizados no notebook de curadoria.

## 1. Configuração do ambiente

As dependências do projeto devem ser instaladas a partir do `requirements.txt` localizado na raiz do repositório.

O notebook também localiza automaticamente a raiz do projeto para permitir a importação dos módulos presentes em `src/`, independentemente de ele ser executado a partir da raiz ou da pasta `notebooks/`.

In [ ]:
from itertools import combinations
from pathlib import Path
import sys

from datasets import load_dataset
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd


def encontrar_raiz_projeto(inicio: Path = Path.cwd()) -> Path:
    """Localiza a raiz do repositório procurando pela pasta src/."""
    for caminho in [inicio, *inicio.parents]:
        if (caminho / "src").is_dir():
            return caminho

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto. "
        "Verifique se a pasta 'src/' existe no repositório."
    )


RAIZ_PROJETO = encontrar_raiz_projeto()

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

print(f"Raiz do projeto: {RAIZ_PROJETO}")

## 2. Carregamento do dataset

O projeto utiliza exclusivamente o dataset público `huggan/wikiart`, disponibilizado no Hugging Face.

O conjunto possui uma única partição original (`train`). As divisões de treino, validação e teste do experimento serão criadas posteriormente, **no nível da pintura completa e antes da geração dos patches**.

In [ ]:
dataset = load_dataset("huggan/wikiart")
train = dataset["train"]

print(dataset)
print(f"Total de obras: {len(train):,}".replace(",", "."))

In [ ]:
print("Features disponíveis:")
display(train.features)

## 3. Construção da tabela de metadados

Para a EDA, os identificadores numéricos de `style`, `artist` e `genre` são convertidos para os nomes definidos pelo próprio dataset.

A coluna `Indice_Dataset` preserva a posição original da obra dentro da partição carregada e será útil para rastreabilidade durante as próximas etapas.

In [ ]:
style_names = train.features["style"].names
artist_names = train.features["artist"].names
genre_names = train.features["genre"].names

df_meta = pd.DataFrame({
    "Indice_Dataset": range(len(train)),
    "style_id": train["style"],
    "artist_id": train["artist"],
    "genre_id": train["genre"],
})

df_meta["Estilo"] = df_meta["style_id"].map(lambda x: style_names[x])
df_meta["Artista"] = df_meta["artist_id"].map(lambda x: artist_names[x])
df_meta["Genero"] = df_meta["genre_id"].map(lambda x: genre_names[x])

display(df_meta.head())

print(f"Estilos: {df_meta['Estilo'].nunique()}")
print(f"Artistas rotulados: {df_meta['Artista'].nunique()}")
print(f"Gêneros: {df_meta['Genero'].nunique()}")

## 4. Distribuição das obras por estilo

A primeira análise verifica quantas obras estão disponíveis em cada um dos 27 rótulos de estilo do WikiArt. Essa distribuição permite avaliar a viabilidade das classes e evidencia o desbalanceamento natural do dataset.

In [ ]:
df_estilos = (
    df_meta["Estilo"]
    .value_counts()
    .rename_axis("Estilo")
    .reset_index(name="Quantidade")
)

display(df_estilos)

In [ ]:
plt.figure(figsize=(10, 9))

plt.barh(
    df_estilos["Estilo"],
    df_estilos["Quantidade"],
)

plt.xlabel("Quantidade de imagens")
plt.ylabel("Estilo artístico")
plt.title("Distribuição de imagens por estilo no WikiArt")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Análise de autoria

A autoria foi analisada para investigar se alguma classe estava excessivamente concentrada em poucos artistas.

Entretanto, o WikiArt possui grande quantidade de registros com `Unknown Artist`. Esse rótulo pode representar várias autorias não identificadas e, portanto, **não deve ser interpretado como um único artista dominante**.

Por esse motivo, a autoria é documentada como característica e limitação dos metadados, mas não será utilizada como principal critério de balanceamento das classes.

In [ ]:
resumo_artistas = (
    df_meta.groupby("Estilo")
    .agg(
        Quantidade_Obras=("Artista", "size"),
        Artistas_Distintos=("Artista", "nunique"),
    )
    .reset_index()
    .sort_values("Quantidade_Obras", ascending=False)
)

display(resumo_artistas)

In [ ]:
artistas_por_estilo = (
    df_meta.groupby(["Estilo", "Artista"])
    .size()
    .reset_index(name="Quantidade")
)

artistas_por_estilo["Percentual"] = (
    artistas_por_estilo.groupby("Estilo")["Quantidade"]
    .transform(lambda valores: 100 * valores / valores.sum())
    .round(2)
)

artistas_por_estilo = artistas_por_estilo.sort_values(
    ["Estilo", "Quantidade"],
    ascending=[True, False],
)

artista_mais_frequente = (
    artistas_por_estilo
    .groupby("Estilo", as_index=False)
    .first()
)

display(
    artista_mais_frequente[
        ["Estilo", "Artista", "Quantidade", "Percentual"]
    ]
)

In [ ]:
unknown_artist_por_estilo = (
    artistas_por_estilo[
        artistas_por_estilo["Artista"] == "Unknown Artist"
    ][["Estilo", "Quantidade", "Percentual"]]
    .sort_values("Percentual", ascending=False)
    .reset_index(drop=True)
)

display(unknown_artist_por_estilo)

### Conclusão sobre autoria

A presença expressiva de `Unknown Artist` reduz a confiabilidade de análises de concentração por autor. Assim, a seleção das macroclasses não será baseada no artista mais frequente. A limitação será registrada na documentação do TCC.

## 6. Distribuição de gêneros nos 27 estilos

O campo `genre` representa o tema ou tipo de pintura (`portrait`, `landscape`, `religious_painting`, entre outros).

Essa variável é especialmente relevante porque pode funcionar como um **atalho temático**: uma rede pode associar determinado conteúdo visual a um movimento artístico sem necessariamente utilizar características estilísticas.

In [ ]:
generos_por_estilo = (
    df_meta.groupby(["Estilo", "Genero"])
    .size()
    .reset_index(name="Quantidade")
)

generos_por_estilo["Percentual"] = (
    generos_por_estilo.groupby("Estilo")["Quantidade"]
    .transform(lambda valores: 100 * valores / valores.sum())
    .round(2)
)

generos_por_estilo = generos_por_estilo.sort_values(
    ["Estilo", "Quantidade"],
    ascending=[True, False],
)

tabela_generos = (
    pd.crosstab(
        df_meta["Estilo"],
        df_meta["Genero"],
        normalize="index",
    )
    * 100
).round(2)

display(generos_por_estilo.head(20))
display(tabela_generos)

## 7. Definição das quatro macroclasses

O escopo do TCC estabelece quatro classes finais. Para preservar a rastreabilidade, o rótulo original `Estilo` continua disponível e uma nova coluna, `Macro_Estilo`, é criada.

### Renaissance
- `Early_Renaissance`
- `High_Renaissance`
- `Northern_Renaissance`
- `Mannerism_Late_Renaissance`

### Cubism
- `Cubism`
- `Analytical_Cubism`
- `Synthetic_Cubism`

### Classes mantidas sem agregação
- `Baroque`
- `Impressionism`

O mapeamento oficial fica centralizado em `src/data/labels.py`, evitando definições divergentes entre notebooks e scripts.

In [ ]:
from src.data.labels import (
    CLASSES_FINAIS,
    ESTILOS_ORIGINAIS_EXPERIMENTO,
    MAPA_MACROCLASSES,
)

df_experimento = df_meta[
    df_meta["Estilo"].isin(ESTILOS_ORIGINAIS_EXPERIMENTO)
].copy()

df_experimento["Macro_Estilo"] = (
    df_experimento["Estilo"]
    .map(MAPA_MACROCLASSES)
)

assert df_experimento["Macro_Estilo"].isna().sum() == 0
assert set(df_experimento["Macro_Estilo"].unique()) == set(CLASSES_FINAIS)

ordem_classes = list(CLASSES_FINAIS)

print("Macroclasses carregadas corretamente.")
display(
    df_experimento["Macro_Estilo"]
    .value_counts()
    .reindex(ordem_classes)
    .rename("Quantidade")
    .to_frame()
)

In [ ]:
composicao_macroclasses = (
    df_experimento
    .groupby(["Macro_Estilo", "Estilo"])
    .size()
    .reset_index(name="Quantidade")
    .sort_values(
        ["Macro_Estilo", "Quantidade"],
        ascending=[True, False],
    )
)

display(composicao_macroclasses)

## 8. Gênero temático por macroclasse

A distribuição é apresentada de duas maneiras:

1. **Distribuição completa**, incluindo `Unknown Genre`.
2. **Distribuição entre gêneros conhecidos**, excluindo `Unknown Genre` antes do cálculo dos percentuais.

Essa separação é importante: por exemplo, dizer que uma porcentagem das obras de uma classe é religiosa considerando todo o conjunto não é equivalente a dizer qual é a participação religiosa apenas entre os registros com gênero conhecido.

In [ ]:
generos_macro = (
    df_experimento
    .groupby(["Macro_Estilo", "Genero"])
    .size()
    .reset_index(name="Quantidade")
)

generos_macro["Percentual_Total"] = (
    generos_macro.groupby("Macro_Estilo")["Quantidade"]
    .transform(lambda valores: 100 * valores / valores.sum())
    .round(2)
)

generos_macro = generos_macro.sort_values(
    ["Macro_Estilo", "Quantidade"],
    ascending=[True, False],
)

tabela_generos_macro = (
    pd.crosstab(
        df_experimento["Macro_Estilo"],
        df_experimento["Genero"],
        normalize="index",
    )
    * 100
).reindex(ordem_classes).round(2)

display(tabela_generos_macro)

In [ ]:
unknown_genero = (
    generos_macro[
        generos_macro["Genero"] == "Unknown Genre"
    ][["Macro_Estilo", "Quantidade", "Percentual_Total"]]
    .set_index("Macro_Estilo")
    .reindex(ordem_classes)
)

display(unknown_genero)

In [ ]:
df_generos_conhecidos = df_experimento[
    df_experimento["Genero"] != "Unknown Genre"
].copy()

tabela_generos_sem_unknown = (
    pd.crosstab(
        df_generos_conhecidos["Macro_Estilo"],
        df_generos_conhecidos["Genero"],
        normalize="index",
    )
    * 100
).reindex(ordem_classes).round(2)

print("Distribuição percentual considerando apenas gêneros conhecidos:")
display(tabela_generos_sem_unknown)

## 9. Visualização da composição temática

O heatmap e o gráfico de barras empilhadas facilitam a comparação entre as macroclasses e evidenciam gêneros com forte associação a movimentos específicos.

In [ ]:
dados_heatmap = tabela_generos_sem_unknown.copy()

fig, ax = plt.subplots(figsize=(13, 5))

imagem = ax.imshow(
    dados_heatmap.values,
    aspect="auto",
)

ax.set_xticks(range(len(dados_heatmap.columns)))
ax.set_xticklabels(
    dados_heatmap.columns,
    rotation=45,
    ha="right",
)

ax.set_yticks(range(len(dados_heatmap.index)))
ax.set_yticklabels(dados_heatmap.index)

ax.set_xlabel("Gênero")
ax.set_ylabel("Movimento artístico")
ax.set_title(
    "Distribuição percentual dos gêneros por movimento artístico"
)

for i in range(dados_heatmap.shape[0]):
    for j in range(dados_heatmap.shape[1]):
        valor = dados_heatmap.iloc[i, j]
        ax.text(
            j,
            i,
            f"{valor:.1f}%",
            ha="center",
            va="center",
            fontsize=8,
        )

fig.colorbar(
    imagem,
    ax=ax,
    label="Percentual (%)",
)

plt.tight_layout()
plt.show()

In [ ]:
tabela_generos_sem_unknown.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
)

plt.title("Composição dos gêneros por movimento artístico")
plt.xlabel("Movimento artístico")
plt.ylabel("Percentual (%)")
plt.xticks(rotation=0)

plt.legend(
    title="Gênero",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
)

plt.tight_layout()
plt.show()

In [ ]:
top3_generos_conhecidos = (
    tabela_generos_sem_unknown
    .stack()
    .rename("Percentual")
    .reset_index()
    .sort_values(
        ["Macro_Estilo", "Percentual"],
        ascending=[True, False],
    )
    .groupby("Macro_Estilo")
    .head(3)
)

display(top3_generos_conhecidos)

## 10. Quantificação das diferenças temáticas

Além da inspeção visual, as diferenças entre as classes são quantificadas em pontos percentuais.

São calculados:

- o maior contraste entre classes para cada gênero;
- as maiores diferenças gênero a gênero entre cada par de macroclasses;
- a distância de variação total entre as distribuições completas, usada como medida comparativa da diferença temática global entre pares de classes.

In [ ]:
comparacao_generos = tabela_generos_sem_unknown.T.copy()

comparacao_generos["Maior_Percentual"] = (
    comparacao_generos[ordem_classes].max(axis=1)
)
comparacao_generos["Classe_Maior"] = (
    comparacao_generos[ordem_classes].idxmax(axis=1)
)
comparacao_generos["Menor_Percentual"] = (
    comparacao_generos[ordem_classes].min(axis=1)
)
comparacao_generos["Classe_Menor"] = (
    comparacao_generos[ordem_classes].idxmin(axis=1)
)
comparacao_generos["Diferenca_pp"] = (
    comparacao_generos["Maior_Percentual"]
    - comparacao_generos["Menor_Percentual"]
).round(2)

comparacao_generos = comparacao_generos.sort_values(
    "Diferenca_pp",
    ascending=False,
)

display(comparacao_generos)

In [ ]:
diferencas_pares = []

for classe_a, classe_b in combinations(ordem_classes, 2):
    for genero in tabela_generos_sem_unknown.columns:
        percentual_a = tabela_generos_sem_unknown.loc[classe_a, genero]
        percentual_b = tabela_generos_sem_unknown.loc[classe_b, genero]

        diferencas_pares.append({
            "Classe_A": classe_a,
            "Classe_B": classe_b,
            "Genero": genero,
            "Percentual_A": percentual_a,
            "Percentual_B": percentual_b,
            "Diferenca_pp": round(abs(percentual_a - percentual_b), 2),
        })

df_diferencas_pares = (
    pd.DataFrame(diferencas_pares)
    .sort_values("Diferenca_pp", ascending=False)
    .reset_index(drop=True)
)

display(df_diferencas_pares.head(15))

In [ ]:
distancias = []

for classe_a, classe_b in combinations(ordem_classes, 2):
    distribuicao_a = tabela_generos_sem_unknown.loc[classe_a] / 100
    distribuicao_b = tabela_generos_sem_unknown.loc[classe_b] / 100

    distancia = 0.5 * (distribuicao_a - distribuicao_b).abs().sum()

    distancias.append({
        "Classe_A": classe_a,
        "Classe_B": classe_b,
        "Distancia_Variacao_Total": round(distancia, 3),
    })

df_distancia_generos = (
    pd.DataFrame(distancias)
    .sort_values("Distancia_Variacao_Total", ascending=False)
    .reset_index(drop=True)
)

display(df_distancia_generos)

## 11. Conclusões da análise exploratória

A EDA identificou pontos centrais para a construção do experimento:

- O WikiArt possui volume suficiente para as quatro macroclasses selecionadas.
- `Unknown Artist` ocorre com frequência relevante e não será tratado como um único autor.
- `Unknown Genre` também representa uma limitação dos metadados e deve ser documentado separadamente.
- A distribuição temática não é homogênea entre as macroclasses.
- O Renascimento apresenta forte predominância de `religious_painting`.
- O Impressionismo apresenta concentração relevante de `landscape`.
- O Cubismo apresenta concentração relevante de `still_life`.
- O Barroco possui distribuição temática comparativamente mais heterogênea.
- Essas diferenças podem favorecer *shortcut learning* e, portanto, não devem ser ignoradas na interpretação do desempenho da rede.

### Decisão metodológica após a EDA

O dataset principal será balanceado **por macroclasse**, utilizando a disponibilidade final do Cubismo como referência e tendo como meta provisória até **2.500 pinturas válidas por classe**.

A distribuição temática natural não será artificialmente igualada no conjunto principal. Em etapa posterior, será realizado um **experimento de controle temático**, inicialmente planejado com obras do gênero `portrait`, presente nas quatro macroclasses.

### Próxima etapa

A próxima etapa será realizada em `02_curadoria_dataset.ipynb` e incluirá:

1. auditoria de integridade das imagens;
2. análise das dimensões;
3. definição do número final de obras por macroclasse;
4. amostragem reproduzível;
5. geração do manifesto do dataset;
6. divisão por pintura em treino, validação e teste.